**Monte Carlo estimation of $\pi$ using the Halton quasi-random sequence.**
The PRNG notebook used a pseudo-random number generator whose points cluster
and leave gaps by chance.
A **quasi-random** (or **low-discrepancy**) sequence is designed to fill
the domain as uniformly as possible while still avoiding a regular grid.

The **Halton sequence** achieves this by writing the index $n$ in a prime
base $p$, reversing the digits, and placing them after the decimal point.
For example in base 2:

$$n=1\to 0.1_2=0.5,\quad
n=2\to 0.01_2=0.25,\quad
n=3\to 0.11_2=0.75,\quad
n=4\to 0.001_2=0.125,\;\ldots$$

The sequence fills $[0,1)$ without clustering.
Using **different prime bases** for each dimension (base 2 for $x$, base 3 for $y$)
ensures the two coordinates are uncorrelated.

**The payoff:** with only 25,600 points (one quarter of the 102,400 used in
the PRNG notebook) the Halton sequence achieves $\approx 0.006\%$ error,
compared to $\approx 0.092\%$ for the PRNG - roughly **15× more accurate
with 4× fewer points**.
This is why quasi-random sequences are preferred over pseudo-random
generators for numerical integration.


**Implementing the Halton sequence with Numba.**
The algorithm iteratively extracts the base-$p$ digits of $n$ from right to left,
accumulating them as fractional contributions $f = 1/p,\,1/p^2,\,1/p^3,\ldots$
The `@vectorize` decorator compiles the scalar function into a ufunc that
accepts NumPy arrays directly, applying the computation element-wise.

---
The `halton` function is JIT-compiled with **Numba** (`@vectorize`) so it
runs at near-C speed despite being written in Python.

In [ ]:
"""mc_circle_halton.ipynb"""

# Cell 01 - Numba-compiled Halton quasi-random sequence
# Reverses the base-p digits of n to produce a value in [0, 1)

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from matplotlib.patches import Circle, Rectangle
from numba import float64, int64, vectorize


@vectorize([float64(int64, int64)], nopython=True)
def halton(n, prime):
    h, f = 0, 1
    while n > 0:
        f = f / prime  # next fractional weight: 1/p, 1/p^2, ...
        h += (n % prime) * f  # add the current base-p digit * its weight
        n = int(n / prime)  # shift right in base p
    return h


# Verify: base-2 Halton should interleave [0,1) evenly
print("Halton base-2 check (n=1..8):")
print(halton(np.arange(1, 9, dtype=np.int64), np.full(8, 2, dtype=np.int64)))

**Sampling region.**
Same $[-1,1]\times[-1,1]$ bounding box as before.
Note `total_dots = 25,600` - intentionally one quarter of the 102,400
used in the PRNG and grid notebooks, to demonstrate that the Halton
sequence achieves better accuracy with fewer points.

In [ ]:
# Cell 02 - Define the sampling region

bbox = Rectangle((-1, -1), 2, 2).get_bbox()  # [-1, 1] x [-1, 1] square
print(bbox)

total_dots = 25_600  # 1/4 of the PRNG/grid notebooks - intentionally smaller
print(f"{total_dots = :,}")

**Generating the Halton sample points.**
$x$ uses base-2 Halton values and $y$ uses base-3 Halton values.
Using coprime bases ensures the two coordinate sequences are
mathematically independent - they do not repeat patterns together.
The interval flip $(1 - h)$ converts $[0,1)$ to $(0,1]$ so that
boundary points contribute to the area count.

In [ ]:
# Cell 03 - Generate Halton sample points in the bounding box
# x uses base 2, y uses base 3 - different primes ensure independence

n_arr = np.arange(total_dots, dtype=np.int64)

x = (1 - halton(n_arr, np.full(total_dots, 2, dtype=np.int64))) * bbox.width + bbox.x0
y = (1 - halton(n_arr, np.full(total_dots, 3, dtype=np.int64))) * bbox.height + bbox.y0

pd.DataFrame({"x": x[:5], "y": y[:5]})

**Distance from the origin.**
Identical vectorized computation as the PRNG and grid notebooks.

In [ ]:
# Cell 04 - Euclidean distance from the origin for every sample point

d = np.sqrt(x**2 + y**2)

pd.DataFrame({"x": x[:5], "y": y[:5], "d": d[:5]})

**Classifying points as inside or outside the circle.**
Same boolean indexing as the other notebooks.

In [ ]:
# Cell 05 - Partition points: inside (d <= 1) vs outside (d > 1)

x_in = x[d <= 1.0]
y_in = y[d <= 1.0]
x_out = x[d > 1.0]
y_out = y[d > 1.0]

pd.DataFrame(
    {"x_in": x_in[:5], "y_in": y_in[:5], "x_out": x_out[:5], "y_out": y_out[:5]}
)

**Scatter plot of the Halton sample.**
Unlike the PRNG scatter, the Halton points show no obvious clustering or gaps.
Unlike the grid scatter, there is no regular lattice structure.
The distribution looks random but is more uniform than true randomness -
this is the defining visual signature of a low-discrepancy sequence.

In [ ]:
# Cell 06 - Scatter plot: red = inside, blue = outside, black circle = exact boundary

fig, ax = plt.subplots(figsize=(7, 7))
fig.canvas.manager.set_window_title("mc_circle_halton")
ax.scatter(x_in, y_in, color="red", marker=".", s=0.5)
ax.scatter(x_out, y_out, color="blue", marker=".", s=0.5)
ax.add_patch(Circle((0, 0), 1, fill=False, color="black", linewidth=1.5))
ax.set_aspect("equal")
ax.set_title(f"Halton Quasi-Random ({total_dots:,} points)")
ax.set_xlabel("x")
ax.set_ylabel("y")
plt.tight_layout()
plt.show()

**Estimating $\pi$ and comparing all four methods.**
The Halton result at 25,600 points should substantially outperform
the PRNG result at 102,400 points,
demonstrating that quasi-random sampling is more efficient than
pseudo-random sampling for numerical integration.

In [ ]:
# Cell 07 - Compute the pi estimate and compare all methods

act = np.pi
est = (bbox.width * bbox.height) * np.count_nonzero(d <= 1.0) / total_dots
err = np.abs((est - act) / act)

print(f"dots total  : {total_dots:,}")
print(f"dots inside : {np.count_nonzero(d <= 1.0):,}")
print(f"actual pi   : {act:.6f}")
print(f"estimated   : {est:.6f}")
print(f"abs error   : {err:.5%}")
print()
print("Method comparison (all approx. same or fewer points):")
print("  PRNG pseudo-random  (N=102,400) : 0.09230%")
print("  Midpoint grid       (N=102,400) : 0.03386%")
print(f"  Halton quasi-random (N= 25,600) : {err:.5%}   <- 4x fewer points")